In [4]:
import geopandas as gpd
import fiona

RAW_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\NUTS_RG_03M_2024_4326.gpkg"  # <-- your downloaded file
OUT_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\europe_admin_clean_03m.gpkg"

layers = fiona.listlayers(RAW_GPKG)
layer_name = layers[0]
gdf = gpd.read_file(RAW_GPKG, layer=layer_name).to_crs("EPSG:4326")

TARGET_CNTR = {"FR", "BE", "IT", "LU"}
gdf = gdf[gdf["CNTR_CODE"].isin(TARGET_CNTR) & gdf["LEVL_CODE"].isin([2, 3])].copy()

# make geometries valid (important!)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
try:
    gdf["geometry"] = gdf.geometry.make_valid()
except Exception:
    gdf["geometry"] = gdf.buffer(0)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

name_col = "NAME_LATN" if "NAME_LATN" in gdf.columns else "NAME_ENGL"
country_map = {"FR":"France","BE":"Belgium","IT":"Italy","LU":"Luxembourg"}
gdf["country"] = gdf["CNTR_CODE"].map(country_map)

regions = (gdf[gdf["LEVL_CODE"]==2]
           .rename(columns={"CNTR_CODE":"country_code","NUTS_ID":"region_code",name_col:"region_name"})
          )[["country","country_code","region_code","region_name","geometry"]]

depts = (gdf[gdf["LEVL_CODE"]==3]
         .rename(columns={"CNTR_CODE":"country_code","NUTS_ID":"dept_code",name_col:"dept_name"})
        )[["country","country_code","dept_code","dept_name","geometry"]]

regions.to_file(OUT_GPKG, layer="regions_nuts2", driver="GPKG")
depts.to_file(OUT_GPKG, layer="departments_nuts3", driver="GPKG")

print("Saved:", OUT_GPKG)
print("Layers:", fiona.listlayers(OUT_GPKG))
print("regions:", regions.shape, "depts:", depts.shape)

Saved: D:\M2_MoSEF\Acclim\DataCollection\data\europe_admin_clean_03m.gpkg
Layers: ['regions_nuts2', 'departments_nuts3']
regions: (60, 5) depts: (253, 5)


In [2]:
import fiona
import geopandas as gpd

ADMIN_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\europe_admin_clean.gpkg"
print(fiona.listlayers(ADMIN_GPKG))

r = gpd.read_file(ADMIN_GPKG, layer="regions_nuts2")
d = gpd.read_file(ADMIN_GPKG, layer="departments_nuts3")
print(r.shape, r.crs)
print(d.shape, d.crs)

['regions_nuts2', 'departments_nuts3']
(60, 5) EPSG:4326
(253, 5) EPSG:4326


In [3]:
import pandas as pd

xlsx10 = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_country_region_department\events_country_region_department_2015_2024_clean10M.xlsx"
comm_long = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_inseecom\events_communes_long_all_years.csv"

summary_depts = pd.read_excel(xlsx10, sheet_name="summary_event_depts")
comm = pd.read_csv(comm_long, dtype=str)

# Events that the commune run says affected France
fr_comm_events = set(comm["event_id"].unique())

# Events where NUTS script had any candidates at all
nuts_candidate_events = set(summary_depts.loc[summary_depts["n_candidates_tested"] > 0, "event_id"])

print("France events (commune):", len(fr_comm_events))
print("NUTS events with candidates:", len(nuts_candidate_events))
print("France events that had NUTS candidates:", len(fr_comm_events & nuts_candidate_events))
print("France events with NO NUTS candidates:", len(fr_comm_events - nuts_candidate_events))

France events (commune): 135
NUTS events with candidates: 597
France events that had NUTS candidates: 135
France events with NO NUTS candidates: 0


In [9]:
import os, glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask

FLOODS_BASE = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps"

# Commune outputs you already have
COMM_LONG = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_inseecom\events_communes_long_all_years.csv"
COMMUNES_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
COMMUNES_LAYER = "commune"
COMMUNE_CODE_COL = "insee_com"

# NUTS clean file (03M recommended)
NUTS_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\europe_admin_clean_03m.gpkg"
DEPT_LAYER = "departments_nuts3"

IGNORE_VALUES = {9999}
THRESHOLD_CM = 0.0
ALL_TOUCHED = True

In [10]:
eid = "2023-02-06---2023-02-13_cluster_15"

In [12]:
comm = pd.read_csv(COMM_LONG, dtype=str)
comm["commune_code"] = comm["commune_code"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
comm["max_cm"] = comm["max_cm"].astype(float)

comm_e = comm[comm["event_id"] == eid].copy()
if comm_e.empty:
    raise ValueError(f"No commune rows found for event_id={eid} in {COMM_LONG}")

top = comm_e.sort_values("max_cm", ascending=False).iloc[0]
print("Top commune for event:", top["commune"], top["commune_code"], "max_cm_commune_output=", top["max_cm"])

communes = gpd.read_file(COMMUNES_GPKG, layer=COMMUNES_LAYER)[[COMMUNE_CODE_COL, "geometry"]].copy()
communes[COMMUNE_CODE_COL] = communes[COMMUNE_CODE_COL].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
communes = communes.to_crs("EPSG:4326")

cgeom = communes.loc[communes[COMMUNE_CODE_COL] == top["commune_code"], "geometry"]
if cgeom.empty:
    raise ValueError("Commune geometry not found for code " + top["commune_code"])
cgeom = cgeom.iloc[0]

Top commune for event: Échigey 21242 max_cm_commune_output= 10.0


In [14]:
# ============================================================
# TEST: commune flood pixel >0 but NUTS3 dept pixel ==0 ?
# This proves whether boundary/generalization is causing NUTS misses.
#
# INPUTS:
# - events_communes_long_all_years.csv (from your France commune run)
# - AdminExpress communes gpkg (France)
# - europe_admin_clean_03m.gpkg (NUTS3 depts)
# - JRC flood rasters in *_filtered folders
#
# OUTPUT:
# - Finds the top commune for a chosen event_id
# - Finds ALL tiles for that event robustly (handles cluster padding + duration)
# - Picks a tile that actually contains flood inside the commune (if possible)
# - Computes max_cm inside commune geometry and inside containing NUTS3 dept geometry
# ============================================================

import os
import re
import glob

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask


# -------------------------
# CONFIG: paths
# -------------------------
FLOODS_BASE = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps"

COMM_LONG = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_inseecom\events_communes_long_all_years.csv"
COMMUNES_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
COMMUNES_LAYER = "commune"
COMMUNE_CODE_COL = "insee_com"   # in gpkg
COMMUNE_NAME_COL = "nom"         # in gpkg

NUTS_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\europe_admin_clean_03m.gpkg"
DEPT_LAYER = "departments_nuts3"

# Masking options (same as your pipeline)
IGNORE_VALUES = {9999}
THRESHOLD_CM = 0.0
ALL_TOUCHED = True


# -------------------------
# HELPERS
# -------------------------
def parse_event_id_components(eid: str):
    """
    eid example: 2023-02-06---2023-02-13_cluster_15
    """
    m = re.match(r"(?P<start>\d{4}-\d{2}-\d{2})---(?P<end>\d{4}-\d{2}-\d{2})_cluster_(?P<cluster>\d+)", eid)
    if not m:
        raise ValueError(f"Bad event_id format: {eid}")
    return m.group("start"), m.group("end"), int(m.group("cluster"))


def find_tiles_for_event(FLOODS_BASE: str, eid: str):
    """
    Robust tile finder:
    - filename contains duration and cluster
    - cluster may be zero-padded to 3 digits (015)
    Searches inside *_filtered folders.
    """
    start, end, cluster = parse_event_id_components(eid)
    cluster3 = f"{cluster:03d}"

    patterns = [
        os.path.join(FLOODS_BASE, "*_filtered",
                     f"WD_MERGE_{start}---{end}_duration_*_days_cluster_{cluster}_A0_*.tif"),
        os.path.join(FLOODS_BASE, "*_filtered",
                     f"WD_MERGE_{start}---{end}_duration_*_days_cluster_{cluster3}_A0_*.tif"),
    ]

    tiles = []
    for p in patterns:
        tiles.extend(glob.glob(p))
    return sorted(set(tiles)), patterns


def max_on_geom(tile_fp: str, geom_wgs84):
    """
    Returns (max_cm or None, n_pixels_used)
    """
    with rasterio.open(tile_fp) as src:
        g = gpd.GeoSeries([geom_wgs84], crs="EPSG:4326").to_crs(src.crs).iloc[0]
        data, _ = mask(
            src,
            [g],
            crop=True,
            filled=False,
            all_touched=ALL_TOUCHED,
            indexes=1
        )

    arr = data[0].astype(np.float32, copy=False)
    vals = arr.compressed() if np.ma.isMaskedArray(arr) else arr.ravel()

    if vals.size == 0:
        return None, 0

    for v in IGNORE_VALUES:
        vals = vals[vals != v]

    if vals.size == 0:
        return None, 0

    return float(vals.max()), int(vals.size)


def pick_tile_with_commune_flood(tiles, commune_geom_wgs84):
    """
    Prefer a tile where commune max_cm > 0, otherwise return first tile.
    """
    best = None
    best_max = -1.0
    for fp in tiles:
        mc, _ = max_on_geom(fp, commune_geom_wgs84)
        if mc is not None and mc > best_max:
            best = fp
            best_max = mc
        if mc is not None and mc > 0:
            return fp, mc
    return (tiles[0] if tiles else None), (best_max if best_max >= 0 else None)


# -------------------------
# MAIN TEST
# -------------------------
# 1) Choose an event_id that is missing in NUTS (example)
eid = "2023-02-06---2023-02-13_cluster_15"
print("Testing event_id:", eid)

# 2) Load commune event table and pick the strongest commune for that event
comm = pd.read_csv(COMM_LONG, dtype=str)
comm["commune_code"] = comm["commune_code"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
comm["max_cm"] = comm["max_cm"].astype(float)

comm_e = comm[comm["event_id"] == eid].copy()
if comm_e.empty:
    raise ValueError(f"No commune rows found for event_id={eid} in {COMM_LONG}")

top = comm_e.sort_values("max_cm", ascending=False).iloc[0]
top_code = top["commune_code"]
top_name = top["commune"]
top_max_from_csv = float(top["max_cm"])
print(f"Top commune from CSV: {top_name} ({top_code}) max_cm={top_max_from_csv}")

# 3) Load commune geometry from AdminExpress (France)
communes = gpd.read_file(COMMUNES_GPKG, layer=COMMUNES_LAYER)[[COMMUNE_CODE_COL, COMMUNE_NAME_COL, "geometry"]].copy()
communes[COMMUNE_CODE_COL] = communes[COMMUNE_CODE_COL].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
communes = communes.to_crs("EPSG:4326")

cgeom = communes.loc[communes[COMMUNE_CODE_COL] == top_code, "geometry"]
if cgeom.empty:
    raise ValueError(f"Commune geometry not found for code={top_code} in {COMMUNES_GPKG}")
cgeom = cgeom.iloc[0]

# 4) Find raster tiles for that event (robust)
tiles, patterns = find_tiles_for_event(FLOODS_BASE, eid)
print("Patterns tried:")
for p in patterns:
    print(" -", p)
print("Tiles found:", len(tiles))
if not tiles:
    raise FileNotFoundError("No raster tile found. Check FLOODS_BASE and event_id/filename conventions.")

# 5) Pick a tile that actually shows flood inside the commune (if possible)
tile_fp, commune_tile_max = pick_tile_with_commune_flood(tiles, cgeom)
print("Selected tile:", tile_fp)
print("Commune max on selected tile (quick scan):", commune_tile_max)

# 6) Load NUTS3 depts and find dept that contains the commune (via representative point)
depts = gpd.read_file(NUTS_GPKG, layer=DEPT_LAYER).to_crs("EPSG:4326")

comm_pt = gpd.GeoDataFrame(
    {"commune_code": [top_code]},
    geometry=[gpd.GeoSeries([cgeom], crs="EPSG:4326").iloc[0].representative_point()],
    crs="EPSG:4326"
)

j = gpd.sjoin(
    comm_pt,
    depts[["dept_code", "dept_name", "country", "country_code", "geometry"]],
    predicate="intersects",
    how="left"
)

print("\nCommune -> NUTS3 mapping:")
print(j[["commune_code", "dept_code", "dept_name", "country", "country_code"]].to_string(index=False))

if pd.isna(j.iloc[0]["dept_code"]):
    raise ValueError("Could not map commune to a NUTS3 dept. Unexpected with correct NUTS file.")

dept_code = j.iloc[0]["dept_code"]
dgeom = depts.loc[depts["dept_code"] == dept_code, "geometry"].iloc[0]

# 7) Compute max on commune vs dept on the SAME tile
max_comm, n_comm = max_on_geom(tile_fp, cgeom)
max_dept, n_dept = max_on_geom(tile_fp, dgeom)

print("\nRESULTS on the SAME tile:")
print(f"Commune mask: max_cm={max_comm}  n_pixels={n_comm}")
print(f"Dept mask:    max_cm={max_dept}  n_pixels={n_dept}")

print("\nAffected? (threshold =", THRESHOLD_CM, ")")
print("Commune affected:", (max_comm is not None and max_comm > THRESHOLD_CM))
print("Dept affected:", (max_dept is not None and max_dept > THRESHOLD_CM))

# 8) If commune is affected but dept isn't, you've proven the mismatch
if (max_comm is not None and max_comm > THRESHOLD_CM) and not (max_dept is not None and max_dept > THRESHOLD_CM):
    print("\n✅ Proven: commune has flood pixels >0, but NUTS3 dept does not on this tile.")
    print("This supports the boundary/generalization/grid-alignment explanation for missing events.")
else:
    print("\nℹ️ Either both detect flood, or neither does on this selected tile.")
    print("If commune didn't detect flood on this tile, try another tile from the list (tiles[:10]) and rerun max_on_geom().")

Testing event_id: 2023-02-06---2023-02-13_cluster_15
Top commune from CSV: Échigey (21242) max_cm=10.0
Patterns tried:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\*_filtered\WD_MERGE_2023-02-06---2023-02-13_duration_*_days_cluster_15_A0_*.tif
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\*_filtered\WD_MERGE_2023-02-06---2023-02-13_duration_*_days_cluster_015_A0_*.tif
Tiles found: 1
Selected tile: D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2023_filtered\WD_MERGE_2023-02-06---2023-02-13_duration_7_days_cluster_015_A0_000008_A_000009_lat_04650_lon_00060_size_0145.tif
Commune max on selected tile (quick scan): 10.0

Commune -> NUTS3 mapping:
commune_code dept_code dept_name country country_code
       21242     FRC11 Côte-d’Or  France           FR

RESULTS on the SAME tile:
Commune mask: max_cm=10.0  n_pixels=2
Dept mask:    max_cm=None  n_pixels=0

Affected? (threshold = 0.0 )
Commune affected: True
Dept affected: False

✅ Proven: commu

In [15]:
import rasterio
from pyproj import CRS

raster_path = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\WD_MERGE_2024-01-01---2024-01-29_duration_28_days_cluster_335_A0_003215_A_003552_lat_05418_lon_01690_size_0452.tif"

with rasterio.open(raster_path) as src:
    crs = src.crs                 # rasterio CRS object (or None)
    print("CRS:", crs)            # e.g. "EPSG:4326" or a PROJ string
    print("WKT:\n", crs.to_wkt() if crs else None)

    # Try to extract an EPSG code (can be None if not identifiable)
    epsg = crs.to_epsg() if crs else None
    print("EPSG:", epsg)

    # Extra: show PROJ4 string
    print("PROJ4:", crs.to_proj4() if crs else None)

CRS: EPSG:27704
WKT:
 PROJCS["unknown",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Azimuthal_Equidistant"],PARAMETER["latitude_of_center",53],PARAMETER["longitude_of_center",24],PARAMETER["false_easting",5837287.81977],PARAMETER["false_northing",2121415.69617],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
EPSG: 27704
PROJ4: +proj=aeqd +lat_0=53 +lon_0=24 +x_0=5837287.81977 +y_0=2121415.69617 +datum=WGS84 +units=m +no_defs=True
